In [0]:


from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

CATALOG = "hackathon_ltm"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

print(f"✅ Using catalog: {CATALOG}.{GOLD_SCHEMA}")



df_employee  = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.hrms_employee_dimension")
df_course    = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.lms_course_master")
df_learning  = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_lms_learning_transaction")
df_feedback  = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_training_feedback")
df_skill     = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_skill_readiness")
df_cert      = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.sp_certification")
df_exp       = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_work_experience")

print("✅ All 7 Silver tables loaded.")



# Aggregate work experience per employee (1 row per employee)
df_exp_agg = (
    df_exp
    .groupBy("employee_id")
    .agg(
        F.max("experience_sequence_no").alias("total_prior_jobs"),
        F.round(F.sum("tenure_in_years"), 2).alias("total_prior_exp_years"),
        F.max(F.when(F.col("experience_sequence_no") == 1, F.col("company_name"))).alias("first_company"),
        F.max(F.when(
            F.col("experience_sequence_no") == F.col("experience_sequence_no"),
            F.col("company_name")
        )).alias("latest_prior_company"),
        F.max("is_fresher_record").cast("boolean").alias("is_fresher")
    )
)

df_dim_employee = (
    df_employee
    .select(
        "employee_id",
        "employee_name",
        "email",
        "department",
        "designation",
        "grade",
        "location",
        "employment_type",
        "status",
        "manager_name",
        "joining_date",
        F.col("tenure_years").alias("current_tenure_years"),
        "tenure_band",
        "joining_year",
        "joining_month",
        "is_manager",
        "is_active"
    )
    .join(df_exp_agg, on="employee_id", how="left")
    .withColumn("is_fresher", F.coalesce(F.col("is_fresher"), F.lit(True)))
    .withColumn("_gold_created_ts", F.current_timestamp())
)

(
    df_dim_employee
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_employee")
)

print(f"✅ dim_employee: {df_dim_employee.count()} rows")



df_dim_course = (
    df_course
    .select(
        "course_id",
        "course_name",
        "category",
        "vendor",
        "delivery_mode",
        "duration_hours",
        "difficulty_level"
    )
    .withColumn("_gold_created_ts", F.current_timestamp())
)

(
    df_dim_course
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_course")
)

print(f"✅ dim_course: {df_dim_course.count()} rows")



# Collect min/max dates across all date columns in learning transactions
date_bounds = df_learning.select(
    F.min(F.col("request_date").cast("date")).alias("min_d"),
    F.max(F.col("completion_date").cast("date")).alias("max_d")
).first()

df_dim_date = (
    spark.range(0, (date_bounds["max_d"] - date_bounds["min_d"]).days + 365)
    .select(
        F.expr("date_add('" + str(date_bounds["min_d"]) + "', CAST(id AS INT))").cast("date").alias("date_id")
    )
    .select(
        "date_id",
        F.year("date_id").alias("year"),
        F.quarter("date_id").alias("quarter"),
        F.month("date_id").alias("month"),
        F.date_format("date_id", "MMMM").alias("month_name"),
        F.dayofmonth("date_id").alias("day"),
        F.dayofweek("date_id").alias("day_of_week"),
        F.date_format("date_id", "EEEE").alias("day_name"),
        F.weekofyear("date_id").alias("week_of_year"),
        F.date_format("date_id", "yyyy-MM").alias("year_month"),
        F.concat(F.lit("Q"), F.quarter("date_id"), F.lit("-"), F.year("date_id")).alias("year_quarter"),
        F.when(F.dayofweek("date_id").isin([1, 7]), True).otherwise(False).alias("is_weekend")
    )
    .withColumn("_gold_created_ts", F.current_timestamp())
)

(
    df_dim_date
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_date")
)

print(f"✅ dim_date: {df_dim_date.count()} rows")



df_dim_skill = (
    df_skill
    .select(
        "skill_id",
        "employee_id",
        "primary_skill",
        "secondary_skill",
        "skill_category",
        "skill_combo",
        "skills_declared",
        "skills_verified",
        "skill_readiness_status",
        "is_declared",
        "is_verified",
        "verification_gap"
    )
    .withColumn("_gold_created_ts", F.current_timestamp())
)

(
    df_dim_skill
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_skill")
)

print(f"✅ dim_skill: {df_dim_skill.count()} rows")



df_dim_certification = (
    df_cert
    .select(
        "certification_id",
        "employee_id",
        "course_id",
        "certification_name",
        "vendor",
        "certificate_uploaded",
        "certificate_status",
        "upload_date",
        "verification_date",
        "days_to_verify",
        "upload_month",
        "is_verified"
    )
    .withColumn("_gold_created_ts", F.current_timestamp())
)

(
    df_dim_certification
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_certification")
)

print(f"✅ dim_certification: {df_dim_certification.count()} rows")



# Bring in feedback rating — one feedback per (employee, course)
df_feedback_slim = (
    df_feedback
    .select(
        "feedback_id",
        "employee_id",
        "course_id",
        F.col("rating").alias("feedback_rating"),
        F.col("is_recommended").alias("feedback_recommended"),
        F.col("has_text_feedback"),
        F.col("feedback_text_length"),
        F.col("rating_band").alias("feedback_rating_band"),
        "trainer_name",
        F.col("feedback_date").cast("date").alias("feedback_date"),
        "feedback_year",
        "feedback_month",
        "feedback_quarter"
    )
)

df_fact = (
    df_learning
    .select(
        "learning_id",
        "employee_id",
        "course_id",
        "course_request_id",
        "approval_status",
        "completion_status",
        "is_passed",
        "course_domain",
        F.col("request_date").cast("date").alias("request_date"),
        F.col("approval_date").cast("date").alias("approval_date"),
        F.col("enrollment_date").cast("date").alias("enrollment_date"),
        F.col("start_date").cast("date").alias("start_date"),
        F.col("completion_date").cast("date").alias("completion_date"),
        "score",
        "passing_score",
        "score_gap",
        "days_to_approve",
        "days_to_enroll",
        "days_to_start",
        "days_to_complete",
        "total_learning_days"
    )
    # Derived measures
    .withColumn("is_completed", F.when(F.col("completion_status") == "Completed", True).otherwise(False))
    .withColumn("score_band",
        F.when(F.col("score") >= 90, "Excellent")
         .when(F.col("score") >= 75, "Good")
         .when(F.col("score") >= 60, "Average")
         .otherwise("Below Average")
    )
    # Join feedback
    .join(df_feedback_slim, on=["employee_id", "course_id"], how="left")
    .withColumn("_gold_created_ts", F.current_timestamp())
)

(
    df_fact
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.fact_learning_activity")
)

print(f"✅ fact_learning_activity: {df_fact.count()} rows")



spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360 AS

WITH learning_stats AS (
    SELECT
        employee_id,
        COUNT(learning_id)                                                              AS total_courses,
        SUM(CASE WHEN completion_status = 'Completed' THEN 1 ELSE 0 END)               AS completed_courses,
        SUM(CASE WHEN is_passed = true THEN 1 ELSE 0 END)                              AS passed_courses,
        ROUND(AVG(CASE WHEN is_passed = true THEN score END), 2)                       AS avg_score_passed,
        ROUND(AVG(total_learning_days), 2)                                             AS avg_learning_days,
        ROUND(AVG(days_to_complete), 2)                                                AS avg_days_to_complete
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    GROUP BY employee_id
),

feedback_stats AS (
    SELECT
        employee_id,
        ROUND(AVG(feedback_rating), 2)                                                 AS avg_feedback_rating,
        SUM(CASE WHEN feedback_recommended = true THEN 1 ELSE 0 END)                   AS total_recommended
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    WHERE feedback_rating IS NOT NULL
    GROUP BY employee_id
),

skill_stats AS (
    SELECT
        employee_id,
        COUNT(skill_id)                                                                AS total_skills,
        SUM(CASE WHEN skill_readiness_status = 'Fully Ready' THEN 1 ELSE 0 END)        AS fully_ready_skills,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)                            AS verified_skills
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_skill
    GROUP BY employee_id
),

cert_stats AS (
    SELECT
        employee_id,
        COUNT(certification_id)                                                        AS total_certs,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)                            AS verified_certs
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_certification
    GROUP BY employee_id
)

SELECT
    e.employee_id,
    e.employee_name,
    e.department,
    e.designation,
    e.grade,
    e.location,
    e.manager_name,
    e.current_tenure_years,
    e.is_manager,
    e.is_fresher,
    e.total_prior_exp_years,

    -- KPI 1: Completeness Rate (% courses completed)
    ROUND(COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2)    AS completeness_rate,

    -- KPI 2: Effectiveness Score (avg score on passed courses)
    COALESCE(l.avg_score_passed, 0)                                                    AS effectiveness_score,

    -- KPI 3: Engagement Rating (avg feedback out of 5)
    COALESCE(f.avg_feedback_rating, 0)                                                 AS engagement_rating,

    -- KPI 4: Skill Gap Index (% skills NOT fully ready; higher = more gaps)
    ROUND(100.0 - COALESCE(s.fully_ready_skills * 100.0 / NULLIF(s.total_skills, 0), 0), 2)  AS skill_gap_index,

    -- KPI 5: Certification Count
    COALESCE(c.verified_certs, 0)                                                      AS verified_certs,

    -- Supporting counts
    COALESCE(l.total_courses, 0)                                                       AS total_courses,
    COALESCE(l.completed_courses, 0)                                                   AS completed_courses,
    COALESCE(l.passed_courses, 0)                                                      AS passed_courses,
    COALESCE(s.total_skills, 0)                                                        AS total_skills,
    COALESCE(s.fully_ready_skills, 0)                                                  AS fully_ready_skills,
    COALESCE(s.verified_skills, 0)                                                     AS verified_skills,
    COALESCE(c.total_certs, 0)                                                         AS total_certs,
    COALESCE(l.avg_learning_days, 0)                                                   AS avg_learning_days,
    COALESCE(l.avg_days_to_complete, 0)                                                AS avg_days_to_complete,
    COALESCE(f.total_recommended, 0)                                                   AS courses_recommended,

    -- Voucher Eligibility (business rule encoded for agent)
    CASE
        WHEN COALESCE(c.verified_certs, 0) = 0                                         THEN '100% Free — No Prior Certs'
        WHEN COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0) >= 85
         AND COALESCE(l.avg_score_passed, 0) >= 80
         AND COALESCE(f.avg_feedback_rating, 0) >= 4.0                                 THEN '100% Free — High Performer'
        WHEN COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0) >= 60 THEN '50% Discount'
        ELSE 'Not Eligible'
    END                                                                                 AS voucher_eligibility,

    CURRENT_TIMESTAMP()                                                                AS _view_refreshed_ts

FROM {CATALOG}.{GOLD_SCHEMA}.dim_employee e
LEFT JOIN learning_stats  l ON e.employee_id = l.employee_id
LEFT JOIN feedback_stats  f ON e.employee_id = f.employee_id
LEFT JOIN skill_stats     s ON e.employee_id = s.employee_id
LEFT JOIN cert_stats      c ON e.employee_id = c.employee_id
""")

print("✅ vw_employee_kpi_360 created")


spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_department_summary AS

SELECT
    e.department,
    COUNT(DISTINCT e.employee_id)                                   AS total_employees,
    ROUND(AVG(k.completeness_rate), 2)                              AS avg_completeness_rate,
    ROUND(AVG(k.effectiveness_score), 2)                            AS avg_effectiveness_score,
    ROUND(AVG(k.engagement_rating), 2)                              AS avg_engagement_rating,
    ROUND(AVG(k.skill_gap_index), 2)                                AS avg_skill_gap_index,
    SUM(k.verified_certs)                                           AS total_verified_certs,
    SUM(k.total_courses)                                            AS total_courses_enrolled,
    SUM(k.completed_courses)                                        AS total_courses_completed,
    SUM(CASE WHEN k.voucher_eligibility LIKE '100%' THEN 1 ELSE 0 END) AS employees_100pct_voucher,
    SUM(CASE WHEN k.voucher_eligibility = '50% Discount' THEN 1 ELSE 0 END) AS employees_50pct_voucher,
    SUM(CASE WHEN k.voucher_eligibility = 'Not Eligible' THEN 1 ELSE 0 END) AS employees_not_eligible,
    CURRENT_TIMESTAMP()                                             AS _view_refreshed_ts
FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360 k
JOIN {CATALOG}.{GOLD_SCHEMA}.dim_employee e ON k.employee_id = e.employee_id
GROUP BY e.department
""")

print("✅ vw_department_summary created")



spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_course_effectiveness AS

SELECT
    f.course_id,
    c.course_name,
    c.category,
    c.vendor,
    c.delivery_mode,
    c.duration_hours,
    c.difficulty_level,
    COUNT(DISTINCT f.employee_id)                                                        AS total_learners,
    SUM(CASE WHEN f.is_completed = true THEN 1 ELSE 0 END)                               AS total_completed,
    ROUND(SUM(CASE WHEN f.is_completed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS completion_rate_pct,
    SUM(CASE WHEN f.is_passed = true THEN 1 ELSE 0 END)                                  AS total_passed,
    ROUND(SUM(CASE WHEN f.is_passed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)    AS pass_rate_pct,
    ROUND(AVG(CASE WHEN f.is_passed = true THEN f.score END), 2)                         AS avg_score_passed,
    ROUND(AVG(f.feedback_rating), 2)                                                     AS avg_feedback_rating,
    ROUND(AVG(f.days_to_complete), 2)                                                    AS avg_days_to_complete,
    ROUND(AVG(f.total_learning_days), 2)                                                 AS avg_total_learning_days,
    COUNT(DISTINCT f.trainer_name)                                                       AS total_trainers,
    CURRENT_TIMESTAMP()                                                                  AS _view_refreshed_ts
FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity f
JOIN {CATALOG}.{GOLD_SCHEMA}.dim_course c ON f.course_id = c.course_id
GROUP BY
    f.course_id, c.course_name, c.category, c.vendor,
    c.delivery_mode, c.duration_hours, c.difficulty_level
""")

print("✅ vw_course_effectiveness created")



spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_voucher_eligibility AS

SELECT
    employee_id,
    employee_name,
    department,
    designation,
    grade,
    manager_name,
    completeness_rate,
    effectiveness_score,
    engagement_rating,
    verified_certs,
    voucher_eligibility,
    CASE
        WHEN voucher_eligibility LIKE '100%' THEN 1
        WHEN voucher_eligibility = '50% Discount' THEN 2
        ELSE 3
    END                             AS voucher_priority_rank,
    _view_refreshed_ts
FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360
ORDER BY voucher_priority_rank, completeness_rate DESC
""")

print("✅ vw_voucher_eligibility created")


spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_agent_context_full AS

WITH skill_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT primary_skill))    AS skills_list,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT skill_category))   AS skill_categories,
        COUNT(DISTINCT primary_skill)                            AS distinct_skill_count,
        SUM(CASE WHEN skill_readiness_status = 'Fully Ready' THEN 1 ELSE 0 END) AS ready_skill_count
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_skill
    GROUP BY employee_id
),

cert_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT certification_name)) AS certifications_list,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT vendor))              AS cert_vendors,
        COUNT(certification_id)                                     AS total_certs,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)         AS verified_cert_count
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_certification
    GROUP BY employee_id
),

course_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT course_id))           AS enrolled_course_ids,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT course_domain))       AS course_domains
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    WHERE is_completed = true
    GROUP BY employee_id
)

SELECT
    -- Identity
    k.employee_id,
    k.employee_name,
    k.department,
    k.designation,
    k.grade,
    k.location,
    k.manager_name,
    k.is_manager,
    k.is_fresher,
    k.total_prior_exp_years,
    k.current_tenure_years,

    -- KPIs
    k.completeness_rate,
    k.effectiveness_score,
    k.engagement_rating,
    k.skill_gap_index,
    k.verified_certs,

    -- Learning counts
    k.total_courses,
    k.completed_courses,
    k.passed_courses,
    k.avg_learning_days,
    k.avg_days_to_complete,

    -- Skills (aggregated for agent)
    COALESCE(sk.skills_list, 'None')         AS skills,
    COALESCE(sk.skill_categories, 'None')    AS skill_categories,
    COALESCE(sk.distinct_skill_count, 0)     AS distinct_skill_count,
    COALESCE(sk.ready_skill_count, 0)        AS ready_skill_count,

    -- Certifications (aggregated for agent)
    COALESCE(ct.certifications_list, 'None') AS certifications,
    COALESCE(ct.cert_vendors, 'None')        AS cert_vendors,
    COALESCE(ct.total_certs, 0)              AS total_certs,
    COALESCE(ct.verified_cert_count, 0)      AS verified_cert_count,

    -- Completed courses (aggregated for agent)
    COALESCE(ca.enrolled_course_ids, 'None') AS completed_course_ids,
    COALESCE(ca.course_domains, 'None')      AS completed_course_domains,

    -- Voucher
    k.voucher_eligibility,

    -- Free-text agent context block (ready to inject into LLM prompt)
    CONCAT(
        k.employee_name, ' (', k.employee_id, ') is a ', k.designation,
        ' in ', k.department, ' at grade ', k.grade,
        ', reporting to ', k.manager_name, '. ',
        'Tenure: ', ROUND(k.current_tenure_years, 1), ' yrs. ',
        'Completed ', k.completed_courses, '/', k.total_courses, ' courses. ',
        'Avg score: ', k.effectiveness_score, '. ',
        'Engagement: ', k.engagement_rating, '/5. ',
        'Skill gap index: ', k.skill_gap_index, '%. ',
        'Verified certs: ', k.verified_certs, '. ',
        'Top skills: ', COALESCE(sk.skills_list, 'None'), '. ',
        'Certifications: ', COALESCE(ct.certifications_list, 'None'), '. ',
        'Voucher eligibility: ', k.voucher_eligibility, '.'
    )                                        AS agent_context_text,

    k._view_refreshed_ts

FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360  k
LEFT JOIN skill_agg  sk ON k.employee_id = sk.employee_id
LEFT JOIN cert_agg   ct ON k.employee_id = ct.employee_id
LEFT JOIN course_agg ca ON k.employee_id = ca.employee_id
""")

print("✅ vw_agent_context_full created")



print("=" * 60)
print("GOLD LAYER — TABLE & VIEW INVENTORY")
print("=" * 60)

tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").collect()
for t in tables:
    obj_type = "VIEW " if t["isTemporary"] == False and "vw_" in t["tableName"] else "TABLE"
    count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{t['tableName']}").count()
    print(f"  {'📊' if 'fact' in t['tableName'] else '📋' if 'dim' in t['tableName'] else '👁️ '} {t['tableName']:<45} {count:>6} rows")

print("=" * 60)
print("\n🤖 Sample Agent Context (first 2 employees):")
spark.sql(f"""
    SELECT employee_id, employee_name, department, voucher_eligibility, agent_context_text
    FROM {CATALOG}.{GOLD_SCHEMA}.vw_agent_context_full
    LIMIT 2
""").show(truncate=80)



✅ Using catalog: hackathon_ltm.gold
✅ All 7 Silver tables loaded.
✅ dim_employee: 476 rows
✅ dim_course: 490 rows
✅ dim_date: 1922 rows
✅ dim_skill: 454 rows
✅ dim_certification: 400 rows
✅ fact_learning_activity: 660 rows
✅ vw_employee_kpi_360 created
✅ vw_department_summary created
✅ vw_course_effectiveness created
✅ vw_voucher_eligibility created
✅ vw_agent_context_full created
GOLD LAYER — TABLE & VIEW INVENTORY
  📋 dim_certification                                400 rows
  📋 dim_course                                       490 rows
  📋 dim_date                                        1922 rows
  📋 dim_employee                                     476 rows
  📋 dim_skill                                        454 rows
  📊 fact_learning_activity                           660 rows
  👁️  vw_agent_context_full                            476 rows
  👁️  vw_course_effectiveness                           48 rows
  👁️  vw_department_summary                              7 rows
  👁️  vw_employe

In [0]:
%sql
ALTER TABLE hackathon_ltm.gold.dim_employee
ADD COLUMNS (
    valid_from    DATE,
    valid_to      DATE,
    is_current    BOOLEAN,
    scd_version   INT
);

-- Backfill existing rows as version 1 / current
UPDATE hackathon_ltm.gold.dim_employee
SET valid_from  = CURRENT_DATE(),
    valid_to    = CAST('9999-12-31' AS DATE),
    is_current  = true,
    scd_version = 1;

num_affected_rows
476


In [0]:
%sql
ALTER TABLE hackathon_ltm.gold.fact_learning_activity
ADD COLUMNS (_gold_updated_ts TIMESTAMP);

In [0]:

 

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

CATALOG       = "hackathon_ltm"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

print(f"✅ Using catalog: {CATALOG}.{GOLD_SCHEMA}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Read Silver Sources

# COMMAND ----------

df_employee = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.hrms_employee_dimension")
df_course   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.lms_course_master")
df_learning = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_lms_learning_transaction")
df_feedback = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_training_feedback")
df_skill    = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_skill_readiness")
df_cert     = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.sp_certification")
df_exp      = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_work_experience")

print("✅ All 7 Silver tables loaded.")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — DIM_EMPLOYEE with SCD Type 2
# MAGIC
# MAGIC ### Why SCD Type 2 here?
# MAGIC Employee attributes like `designation`, `department`, `grade`, and `manager_name` change
# MAGIC over time (promotions, transfers, reorgs). SCD Type 2 keeps a full history row for each
# MAGIC change so historical KPI analysis stays accurate — e.g. "what was this employee's
# MAGIC completion rate when they were in Engineering vs after they moved to AI & Data Science?"
# MAGIC
# MAGIC ### How it works (batch CDC pattern):
# MAGIC 1. Compare incoming Silver rows against the current Gold dim
# MAGIC 2. Where any tracked column has changed → expire the old row (`valid_to = today`, `is_current = false`)
# MAGIC 3. Insert the new row with `valid_from = today`, `valid_to = 9999-12-31`, `is_current = true`
# MAGIC 4. New employees → insert fresh row directly
# MAGIC
# MAGIC ### SCD Type 2 tracked columns (columns that trigger a new version):
# MAGIC - `designation`, `department`, `grade`, `manager_name`, `location`, `employment_type`, `status`

# COMMAND ----------

# ── Step 1: Aggregate work experience per employee (unchanged) ──
df_exp_agg = (
    df_exp
    .groupBy("employee_id")
    .agg(
        F.max("experience_sequence_no").alias("total_prior_jobs"),
        F.round(F.sum("tenure_in_years"), 2).alias("total_prior_exp_years"),
        F.max(F.when(F.col("experience_sequence_no") == 1, F.col("company_name"))).alias("first_company"),
        F.max("is_fresher_record").cast("boolean").alias("is_fresher")
    )
)

# ── Step 2: Build incoming (new/updated) employee snapshot ──
df_incoming = (
    df_employee
    .select(
        "employee_id",
        "employee_name",
        "email",
        "department",
        "designation",
        "grade",
        "location",
        "employment_type",
        "status",
        "manager_name",
        "joining_date",
        F.col("tenure_years").alias("current_tenure_years"),
        "tenure_band",
        "joining_year",
        "joining_month",
        "is_manager",
        "is_active"
    )
    .join(df_exp_agg, on="employee_id", how="left")
    .withColumn("is_fresher", F.coalesce(F.col("is_fresher"), F.lit(True)))
)

# ── Step 3: First run — create the table with SCD Type 2 columns ──
scd2_table = f"{CATALOG}.{GOLD_SCHEMA}.dim_employee"

if not spark.catalog.tableExists(scd2_table):
    (
        df_incoming
        .withColumn("valid_from",    F.current_date())
        .withColumn("valid_to",      F.lit("9999-12-31").cast("date"))
        .withColumn("is_current",    F.lit(True))
        .withColumn("scd_version",   F.lit(1))
        .withColumn("_gold_created_ts", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(scd2_table)
    )
    print(f"✅ dim_employee (initial load): {df_incoming.count()} rows")

else:
    # ── Step 4: Incremental run — detect changes and apply SCD Type 2 ──

    # SCD Type 2 tracked columns — a change in any of these creates a new version row
    scd2_tracked_cols = ["designation", "department", "grade", "manager_name",
                         "location", "employment_type", "status"]

    dt_employee = DeltaTable.forName(spark, scd2_table)
    existing_df = dt_employee.toDF().filter(F.col("is_current") == True)

    # Identify changed records (any SCD2 column differs)
    change_condition = " OR ".join([
        f"existing.{c} != incoming.{c}" for c in scd2_tracked_cols
    ])

    df_changed = (
        df_incoming.alias("incoming")
        .join(existing_df.alias("existing"), on="employee_id", how="inner")
        .where(change_condition)
        .select(F.col("incoming.employee_id"))
    )

    changed_ids = [r.employee_id for r in df_changed.collect()]

    if changed_ids:
        # Expire old rows for changed employees
        dt_employee.update(
            condition = F.col("employee_id").isin(changed_ids) & (F.col("is_current") == True),
            set = {
                "valid_to":   F.current_date(),
                "is_current": F.lit(False)
            }
        )

        # Insert new current rows for changed employees
        df_new_versions = (
            df_incoming
            .filter(F.col("employee_id").isin(changed_ids))
            .withColumn("valid_from",  F.current_date())
            .withColumn("valid_to",    F.lit("9999-12-31").cast("date"))
            .withColumn("is_current",  F.lit(True))
            .withColumn("scd_version",
                # Increment version number from the last version for this employee
                F.lit(1) + F.coalesce(
                    existing_df.filter(F.col("employee_id").isin(changed_ids))
                               .groupBy("employee_id")
                               .agg(F.max("scd_version").alias("max_v"))
                               .select("max_v").first()[0],
                    0
                )
            )
            .withColumn("_gold_created_ts", F.current_timestamp())
        )
        df_new_versions.write.format("delta").mode("append").saveAsTable(scd2_table)
        print(f"✅ dim_employee SCD2: {len(changed_ids)} employees versioned")

    # Upsert net-new employees (CDC INSERT pattern)
    df_new_employees = (
        df_incoming.alias("incoming")
        .join(existing_df.select("employee_id").alias("existing"),
              on="employee_id", how="left_anti")
        .withColumn("valid_from",  F.current_date())
        .withColumn("valid_to",    F.lit("9999-12-31").cast("date"))
        .withColumn("is_current",  F.lit(True))
        .withColumn("scd_version", F.lit(1))
        .withColumn("_gold_created_ts", F.current_timestamp())
    )
    if df_new_employees.count() > 0:
        df_new_employees.write.format("delta").mode("append").saveAsTable(scd2_table)
        print(f"✅ dim_employee CDC INSERT: {df_new_employees.count()} new employees added")
    else:
        print("✅ dim_employee: no new employees to insert")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — DIM_COURSE (SCD Type 1 MERGE)
# MAGIC
# MAGIC Course attributes rarely change structurally, but vendor, difficulty level, or duration
# MAGIC may be updated. SCD Type 1 MERGE simply overwrites — no history needed for course metadata.

# COMMAND ----------

df_dim_course = (
    df_course
    .select(
        "course_id",
        "course_name",
        "category",
        "vendor",
        "delivery_mode",
        "duration_hours",
        "difficulty_level"
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

course_table = f"{CATALOG}.{GOLD_SCHEMA}.dim_course"

if not spark.catalog.tableExists(course_table):
    df_dim_course.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(course_table)
else:
    # Add _gold_updated_ts column if it doesn't exist (schema evolution)
    existing_cols = spark.table(course_table).columns
    if "_gold_updated_ts" not in existing_cols:
        spark.sql(f"ALTER TABLE {course_table} ADD COLUMNS (_gold_updated_ts TIMESTAMP)")
    
    DeltaTable.forName(spark, course_table).alias("tgt").merge(
        df_dim_course.alias("src"), "tgt.course_id = src.course_id"
    ).whenMatchedUpdate(set = {
        "course_name": "src.course_name",
        "category": "src.category",
        "vendor": "src.vendor",
        "delivery_mode": "src.delivery_mode",
        "duration_hours": "src.duration_hours",
        "difficulty_level": "src.difficulty_level",
        "_gold_updated_ts": "src._gold_updated_ts"
    }).whenNotMatchedInsert(values = {
        "course_id": "src.course_id",
        "course_name": "src.course_name",
        "category": "src.category",
        "vendor": "src.vendor",
        "delivery_mode": "src.delivery_mode",
        "duration_hours": "src.duration_hours",
        "difficulty_level": "src.difficulty_level",
        "_gold_created_ts": "current_timestamp()",
        "_gold_updated_ts": "src._gold_updated_ts"
    }).execute()

print(f"✅ dim_course: {spark.table(course_table).count()} rows")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — DIM_DATE (static overwrite — date spine never changes)

# COMMAND ----------

date_bounds = df_learning.select(
    F.min(F.col("request_date").cast("date")).alias("min_d"),
    F.max(F.col("completion_date").cast("date")).alias("max_d")
).first()

df_dim_date = (
    spark.range(0, (date_bounds["max_d"] - date_bounds["min_d"]).days + 365)
    .select(
        F.expr("date_add('" + str(date_bounds["min_d"]) + "', CAST(id AS INT))").cast("date").alias("date_id")
    )
    .select(
        "date_id",
        F.year("date_id").alias("year"),
        F.quarter("date_id").alias("quarter"),
        F.month("date_id").alias("month"),
        F.date_format("date_id", "MMMM").alias("month_name"),
        F.dayofmonth("date_id").alias("day"),
        F.dayofweek("date_id").alias("day_of_week"),
        F.date_format("date_id", "EEEE").alias("day_name"),
        F.weekofyear("date_id").alias("week_of_year"),
        F.date_format("date_id", "yyyy-MM").alias("year_month"),
        F.concat(F.lit("Q"), F.quarter("date_id"), F.lit("-"), F.year("date_id")).alias("year_quarter"),
        F.when(F.dayofweek("date_id").isin([1, 7]), True).otherwise(False).alias("is_weekend")
    )
    .withColumn("_gold_created_ts", F.current_timestamp())
)

(
    df_dim_date.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_date")
)
print(f"✅ dim_date: {df_dim_date.count()} rows")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — DIM_SKILL (SCD Type 1 MERGE)
# MAGIC
# MAGIC Skill readiness status changes as employees get verified — overwrite in place is correct.

# COMMAND ----------

df_dim_skill = (
    df_skill
    .select(
        "skill_id", "employee_id", "primary_skill", "secondary_skill",
        "skill_category", "skill_combo", "skills_declared", "skills_verified",
        "skill_readiness_status", "is_declared", "is_verified", "verification_gap"
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

skill_table = f"{CATALOG}.{GOLD_SCHEMA}.dim_skill"

if not spark.catalog.tableExists(skill_table):
    df_dim_skill.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(skill_table)
else:
    # Add _gold_updated_ts column if it doesn't exist (schema evolution)
    existing_cols = spark.table(skill_table).columns
    if "_gold_updated_ts" not in existing_cols:
        spark.sql(f"ALTER TABLE {skill_table} ADD COLUMNS (_gold_updated_ts TIMESTAMP)")
    
    DeltaTable.forName(spark, skill_table).alias("tgt").merge(
        df_dim_skill.alias("src"), "tgt.skill_id = src.skill_id AND tgt.employee_id = src.employee_id"
    ).whenMatchedUpdate(set = {
        "primary_skill": "src.primary_skill",
        "secondary_skill": "src.secondary_skill",
        "skill_category": "src.skill_category",
        "skill_combo": "src.skill_combo",
        "skills_declared": "src.skills_declared",
        "skills_verified": "src.skills_verified",
        "skill_readiness_status": "src.skill_readiness_status",
        "is_declared": "src.is_declared",
        "is_verified": "src.is_verified",
        "verification_gap": "src.verification_gap",
        "_gold_updated_ts": "src._gold_updated_ts"
    }).whenNotMatchedInsert(values = {
        "skill_id": "src.skill_id",
        "employee_id": "src.employee_id",
        "primary_skill": "src.primary_skill",
        "secondary_skill": "src.secondary_skill",
        "skill_category": "src.skill_category",
        "skill_combo": "src.skill_combo",
        "skills_declared": "src.skills_declared",
        "skills_verified": "src.skills_verified",
        "skill_readiness_status": "src.skill_readiness_status",
        "is_declared": "src.is_declared",
        "is_verified": "src.is_verified",
        "verification_gap": "src.verification_gap",
        "_gold_created_ts": "current_timestamp()",
        "_gold_updated_ts": "src._gold_updated_ts"
    }).execute()

print(f"✅ dim_skill: {spark.table(skill_table).count()} rows")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — DIM_CERTIFICATION (SCD Type 1 MERGE)
# MAGIC
# MAGIC Certification status can flip from 'Pending' to 'Verified'. Merge overwrites the status in place.

# COMMAND ----------

df_dim_certification = (
    df_cert
    .select(
        "certification_id", "employee_id", "course_id", "certification_name",
        "vendor", "certificate_uploaded", "certificate_status", "upload_date",
        "verification_date", "days_to_verify", "upload_month", "is_verified"
    )
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

cert_table = f"{CATALOG}.{GOLD_SCHEMA}.dim_certification"

if not spark.catalog.tableExists(cert_table):
    df_dim_certification.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(cert_table)
else:
    # Add _gold_updated_ts column if it doesn't exist (schema evolution)
    existing_cols = spark.table(cert_table).columns
    if "_gold_updated_ts" not in existing_cols:
        spark.sql(f"ALTER TABLE {cert_table} ADD COLUMNS (_gold_updated_ts TIMESTAMP)")
    
    DeltaTable.forName(spark, cert_table).alias("tgt").merge(
        df_dim_certification.alias("src"), "tgt.certification_id = src.certification_id"
    ).whenMatchedUpdate(set = {
        "employee_id": "src.employee_id",
        "course_id": "src.course_id",
        "certification_name": "src.certification_name",
        "vendor": "src.vendor",
        "certificate_uploaded": "src.certificate_uploaded",
        "certificate_status": "src.certificate_status",
        "upload_date": "src.upload_date",
        "verification_date": "src.verification_date",
        "days_to_verify": "src.days_to_verify",
        "upload_month": "src.upload_month",
        "is_verified": "src.is_verified",
        "_gold_updated_ts": "src._gold_updated_ts"
    }).whenNotMatchedInsert(values = {
        "certification_id": "src.certification_id",
        "employee_id": "src.employee_id",
        "course_id": "src.course_id",
        "certification_name": "src.certification_name",
        "vendor": "src.vendor",
        "certificate_uploaded": "src.certificate_uploaded",
        "certificate_status": "src.certificate_status",
        "upload_date": "src.upload_date",
        "verification_date": "src.verification_date",
        "days_to_verify": "src.days_to_verify",
        "upload_month": "src.upload_month",
        "is_verified": "src.is_verified",
        "_gold_created_ts": "current_timestamp()",
        "_gold_updated_ts": "src._gold_updated_ts"
    }).execute()

print(f"✅ dim_certification: {spark.table(cert_table).count()} rows")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — DIM_VOUCHER_HISTORY (append-only audit log)
# MAGIC
# MAGIC **Purpose:** Tracks every voucher that has ever been issued per employee.
# MAGIC This is what enables the "first-timer" voucher rule — we check this table
# MAGIC to determine if an employee has previously received any voucher.
# MAGIC
# MAGIC **Grain:** One row per voucher event (employee × voucher_date).
# MAGIC On first run this table is empty — it gets populated only when vouchers are
# MAGIC actually issued (by the agent or a downstream process calling `insert_voucher_event()`).

# COMMAND ----------

voucher_hist_table = f"{CATALOG}.{GOLD_SCHEMA}.dim_voucher_history"

if not spark.catalog.tableExists(voucher_hist_table):
    schema = StructType([
        StructField("voucher_id",        StringType(),    False),
        StructField("employee_id",       StringType(),    False),
        StructField("voucher_date",      DateType(),      False),
        StructField("voucher_type",      StringType(),    False),  # '100% Free' | '50% Discount'
        StructField("trigger_rule",      StringType(),    True),   # 'First Timer' | 'High Performer' etc.
        StructField("course_id",         StringType(),    True),   # course this voucher was for
        StructField("issued_by",         StringType(),    True),   # 'agent' | 'manual' | 'system'
        StructField("_gold_created_ts",  TimestampType(), True),
    ])
    empty_df = spark.createDataFrame([], schema)
    (
        empty_df.write.format("delta")
        .mode("overwrite").option("overwriteSchema","true")
        .saveAsTable(voucher_hist_table)
    )
    print("✅ dim_voucher_history: created (empty — populated on voucher issuance)")
else:
    print(f"✅ dim_voucher_history: {spark.table(voucher_hist_table).count()} prior vouchers on record")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — FACT_LEARNING_ACTIVITY (MERGE on learning_id)
# MAGIC
# MAGIC **CDC MERGE pattern:** `learning_id` is the natural key.
# MAGIC - If `learning_id` already exists → update (score, completion_status etc. may change)
# MAGIC - If new → insert
# MAGIC
# MAGIC This avoids reprocessing the entire fact table on every pipeline run.

# COMMAND ----------

df_feedback_slim = (
    df_feedback
    .select(
        "feedback_id", "employee_id", "course_id",
        F.col("rating").alias("feedback_rating"),
        F.col("is_recommended").alias("feedback_recommended"),
        "has_text_feedback", "feedback_text_length",
        F.col("rating_band").alias("feedback_rating_band"),
        "trainer_name",
        F.col("feedback_date").cast("date").alias("feedback_date"),
        "feedback_year", "feedback_month", "feedback_quarter"
    )
)

df_fact_incoming = (
    df_learning
    .select(
        "learning_id", "employee_id", "course_id", "course_request_id",
        "approval_status", "completion_status", "is_passed", "course_domain",
        F.col("request_date").cast("date").alias("request_date"),
        F.col("approval_date").cast("date").alias("approval_date"),
        F.col("enrollment_date").cast("date").alias("enrollment_date"),
        F.col("start_date").cast("date").alias("start_date"),
        F.col("completion_date").cast("date").alias("completion_date"),
        "score", "passing_score", "score_gap",
        "days_to_approve", "days_to_enroll", "days_to_start",
        "days_to_complete", "total_learning_days"
    )
    .withColumn("is_completed",
        F.when(F.col("completion_status") == "Completed", True).otherwise(False))
    .withColumn("score_band",
        F.when(F.col("score") >= 90, "Excellent")
         .when(F.col("score") >= 75, "Good")
         .when(F.col("score") >= 60, "Average")
         .otherwise("Below Average"))
    .join(df_feedback_slim, on=["employee_id", "course_id"], how="left")
    .withColumn("_gold_updated_ts", F.current_timestamp())
)

# Deduplicate by learning_id (keep most recent based on _gold_updated_ts)
window_spec = Window.partitionBy("learning_id").orderBy(F.col("_gold_updated_ts").desc())
df_fact_incoming = df_fact_incoming.withColumn("row_num", F.row_number().over(window_spec)) \
                                    .filter(F.col("row_num") == 1) \
                                    .drop("row_num")

fact_table = f"{CATALOG}.{GOLD_SCHEMA}.fact_learning_activity"

if not spark.catalog.tableExists(fact_table):
    df_fact_incoming.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(fact_table)
    print(f"✅ fact_learning_activity (initial load): {df_fact_incoming.count()} rows")
else:
    # Add _gold_updated_ts column if it doesn't exist (schema evolution)
    existing_cols = spark.table(fact_table).columns
    if "_gold_updated_ts" not in existing_cols:
        spark.sql(f"ALTER TABLE {fact_table} ADD COLUMNS (_gold_updated_ts TIMESTAMP)")
    
    DeltaTable.forName(spark, fact_table).alias("tgt").merge(
        df_fact_incoming.alias("src"), "tgt.learning_id = src.learning_id"
    ).whenMatchedUpdate(set = {
        col: f"src.{col}" for col in df_fact_incoming.columns if col != "learning_id"
    }).whenNotMatchedInsert(values = {
        **{col: f"src.{col}" for col in df_fact_incoming.columns},
        "_gold_created_ts": "current_timestamp()"
    }).execute()
    print(f"✅ fact_learning_activity (MERGE): {spark.table(fact_table).count()} total rows")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 10 — VIEW: vw_employee_kpi_360
# MAGIC
# MAGIC ### KPI 3 — Engagement Score (v2: weighted composite)
# MAGIC
# MAGIC Old logic only used `avg(feedback_rating)` which measures *satisfaction*, not engagement.
# MAGIC The new weighted engagement score combines 5 behavioral signals:
# MAGIC
# MAGIC | Signal | Weight | Rationale |
# MAGIC |---|---|---|
# MAGIC | Completion rate | 30% | Did the employee actually finish courses? |
# MAGIC | Pass rate | 25% | Did they learn enough to pass assessments? |
# MAGIC | Avg score (normalised to /100) | 20% | How well did they perform in assessments? |
# MAGIC | Feedback rating (normalised to /100) | 15% | Did they find the training valuable? |
# MAGIC | Completion speed index | 10% | Did they complete on time vs duration? Faster = more engaged |
# MAGIC
# MAGIC Final score is scaled to 0–100. A score of 0 on any signal means no penalty
# MAGIC if there's no data — COALESCE defaults protect against NULLs.
# MAGIC
# MAGIC ### Voucher Logic (v2)
# MAGIC
# MAGIC New Rule 0 (highest priority — checked first):
# MAGIC **First-Timer Rule** — if the employee has ZERO prior vouchers on record in
# MAGIC `dim_voucher_history` AND has completed at least one course (completion_status = Completed),
# MAGIC they automatically qualify for a 100% Free voucher regardless of score or feedback.
# MAGIC
# MAGIC Logic waterfall (evaluated top-to-bottom, first match wins):
# MAGIC 1. First Timer: no prior vouchers AND at least 1 completed course → **100% Free**
# MAGIC 2. High Performer: completeness ≥ 85% AND effectiveness ≥ 80 AND engagement ≥ 70 → **100% Free**
# MAGIC 3. Good Progress: completeness ≥ 60% → **50% Discount**
# MAGIC 4. Otherwise → **Not Eligible**

# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360 AS

WITH learning_stats AS (
    SELECT
        employee_id,
        COUNT(learning_id)                                                              AS total_courses,
        SUM(CASE WHEN completion_status = 'Completed' THEN 1 ELSE 0 END)               AS completed_courses,
        SUM(CASE WHEN is_passed = true THEN 1 ELSE 0 END)                              AS passed_courses,
        ROUND(AVG(CASE WHEN is_passed = true THEN score END), 2)                       AS avg_score_passed,
        ROUND(AVG(total_learning_days), 2)                                             AS avg_learning_days,
        ROUND(AVG(days_to_complete), 2)                                                AS avg_days_to_complete,
        ROUND(AVG(CASE WHEN is_passed = true THEN score ELSE 0 END), 2)                AS avg_score_all
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    GROUP BY employee_id
),

feedback_stats AS (
    SELECT
        employee_id,
        ROUND(AVG(feedback_rating), 2)                                                 AS avg_feedback_rating,
        SUM(CASE WHEN feedback_recommended = true THEN 1 ELSE 0 END)                   AS total_recommended
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    WHERE feedback_rating IS NOT NULL
    GROUP BY employee_id
),

course_duration_stats AS (
    -- Join with dim_course to get the expected duration per course
    -- completion_speed_index: 1.0 = completed exactly on time,
    --   >1.0 = completed faster (more engaged), capped at 2.0 to avoid outlier inflation
    SELECT
        f.employee_id,
        ROUND(
            AVG(
                LEAST(
                    COALESCE(c.duration_hours, 1) / NULLIF(f.days_to_complete, 0),
                    2.0
                )
            ),
        2) AS avg_completion_speed_index
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity f
    JOIN {CATALOG}.{GOLD_SCHEMA}.dim_course c ON f.course_id = c.course_id
    WHERE f.is_completed = true AND f.days_to_complete > 0
    GROUP BY f.employee_id
),

skill_stats AS (
    SELECT
        employee_id,
        COUNT(skill_id)                                                                AS total_skills,
        SUM(CASE WHEN skill_readiness_status = 'Fully Ready' THEN 1 ELSE 0 END)        AS fully_ready_skills,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)                            AS verified_skills
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_skill
    GROUP BY employee_id
),

cert_stats AS (
    SELECT
        employee_id,
        COUNT(certification_id)                                                        AS total_certs,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)                            AS verified_certs
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_certification
    GROUP BY employee_id
),

-- FIX 3: Weighted engagement score (5 signals)
engagement_calc AS (
    SELECT
        l.employee_id,

        -- Signal 1: Completion rate (0–100)  weight 30%
        ROUND(COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2)
            AS sig_completion_rate,

        -- Signal 2: Pass rate (0–100)  weight 25%
        ROUND(COALESCE(l.passed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2)
            AS sig_pass_rate,

        -- Signal 3: Avg score normalised to 0–100  weight 20%
        ROUND(COALESCE(l.avg_score_all, 0), 2)
            AS sig_avg_score,

        -- Signal 4: Feedback rating normalised to 0–100 (rating is /5, * 20)  weight 15%
        ROUND(COALESCE(f.avg_feedback_rating, 0) * 20.0, 2)
            AS sig_feedback_normalised,

        -- Signal 5: Completion speed index normalised to 0–100 (capped at 2.0, so /2 * 100)  weight 10%
        ROUND(LEAST(COALESCE(cs.avg_completion_speed_index, 1.0), 2.0) / 2.0 * 100.0, 2)
            AS sig_speed_normalised,

        -- Weighted composite score (0–100)
        ROUND(
              COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0)  * 0.30
            + COALESCE(l.passed_courses    * 100.0 / NULLIF(l.total_courses, 0), 0)  * 0.25
            + COALESCE(l.avg_score_all, 0)                                            * 0.20
            + COALESCE(f.avg_feedback_rating, 0) * 20.0                              * 0.15
            + LEAST(COALESCE(cs.avg_completion_speed_index, 1.0), 2.0) / 2.0 * 100.0 * 0.10
        , 2) AS engagement_score
    FROM learning_stats l
    LEFT JOIN feedback_stats f  ON l.employee_id = f.employee_id
    LEFT JOIN course_duration_stats cs ON l.employee_id = cs.employee_id
),

-- FIX 4: First-timer check — has this employee ever received any voucher before?
first_timer_check AS (
    SELECT
        e.employee_id,
        CASE
            WHEN COUNT(v.voucher_id) = 0 THEN true
            ELSE false
        END AS is_first_time_voucher
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_employee e
    LEFT JOIN {CATALOG}.{GOLD_SCHEMA}.dim_voucher_history v
           ON e.employee_id = v.employee_id
    WHERE e.is_current = true
    GROUP BY e.employee_id
)

SELECT
    e.employee_id,
    e.employee_name,
    e.department,
    e.designation,
    e.grade,
    e.location,
    e.manager_name,
    e.current_tenure_years,
    e.is_manager,
    e.is_fresher,
    e.total_prior_exp_years,
    e.valid_from                                                                        AS employee_version_from,
    e.scd_version                                                                       AS employee_scd_version,

    -- KPI 1: Completeness Rate
    ROUND(COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2)    AS completeness_rate,

    -- KPI 2: Effectiveness Score (avg score on passed courses, 0–100)
    COALESCE(l.avg_score_passed, 0)                                                    AS effectiveness_score,

    -- KPI 3: Engagement Score (v2 — weighted composite, 0–100)
    COALESCE(eg.engagement_score, 0)                                                   AS engagement_score,
    COALESCE(eg.sig_completion_rate, 0)                                                AS eng_sig_completion_rate,
    COALESCE(eg.sig_pass_rate, 0)                                                      AS eng_sig_pass_rate,
    COALESCE(eg.sig_avg_score, 0)                                                      AS eng_sig_avg_score,
    COALESCE(eg.sig_feedback_normalised, 0)                                            AS eng_sig_feedback,
    COALESCE(eg.sig_speed_normalised, 0)                                               AS eng_sig_speed,

    -- KPI 4: Skill Gap Index (higher = more gaps)
    ROUND(100.0 - COALESCE(s.fully_ready_skills * 100.0 / NULLIF(s.total_skills, 0), 0), 2) AS skill_gap_index,

    -- KPI 5: Verified Certifications
    COALESCE(c.verified_certs, 0)                                                      AS verified_certs,

    -- Supporting counts
    COALESCE(l.total_courses, 0)                                                       AS total_courses,
    COALESCE(l.completed_courses, 0)                                                   AS completed_courses,
    COALESCE(l.passed_courses, 0)                                                      AS passed_courses,
    COALESCE(s.total_skills, 0)                                                        AS total_skills,
    COALESCE(s.fully_ready_skills, 0)                                                  AS fully_ready_skills,
    COALESCE(s.verified_skills, 0)                                                     AS verified_skills,
    COALESCE(c.total_certs, 0)                                                         AS total_certs,
    COALESCE(l.avg_learning_days, 0)                                                   AS avg_learning_days,
    COALESCE(l.avg_days_to_complete, 0)                                                AS avg_days_to_complete,
    COALESCE(f.total_recommended, 0)                                                   AS courses_recommended,
    ft.is_first_time_voucher,

    -- ── VOUCHER ELIGIBILITY (v2) ──
    -- Rule waterfall — first match wins (top = highest priority)
    CASE
        -- Rule 0: First Timer — zero prior vouchers AND at least 1 completed course
        WHEN ft.is_first_time_voucher = true
         AND COALESCE(l.completed_courses, 0) >= 1
                                                    THEN '100% Free — First Timer'

        -- Rule 1: High Performer — strong completeness + effectiveness + engagement
        WHEN COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0) >= 85
         AND COALESCE(l.avg_score_passed, 0) >= 80
         AND COALESCE(eg.engagement_score, 0) >= 70   THEN '100% Free — High Performer'

        -- Rule 2: Good Progress — reasonable completeness
        WHEN COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0) >= 60
                                                    THEN '50% Discount'

        -- Fallback
        ELSE 'Not Eligible'
    END                                                                                 AS voucher_eligibility,

    CURRENT_TIMESTAMP()                                                                AS _view_refreshed_ts

FROM {CATALOG}.{GOLD_SCHEMA}.dim_employee e
LEFT JOIN learning_stats        l  ON e.employee_id = l.employee_id
LEFT JOIN feedback_stats        f  ON e.employee_id = f.employee_id
LEFT JOIN engagement_calc       eg ON e.employee_id = eg.employee_id
LEFT JOIN skill_stats           s  ON e.employee_id = s.employee_id
LEFT JOIN cert_stats            c  ON e.employee_id = c.employee_id
LEFT JOIN first_timer_check     ft ON e.employee_id = ft.employee_id
WHERE e.is_current = true  -- only current employee version from SCD Type 2
""")

print("✅ vw_employee_kpi_360 (v2) created")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 11 — VIEW: vw_department_summary

# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_department_summary AS

SELECT
    e.department,
    COUNT(DISTINCT e.employee_id)                                           AS total_employees,
    ROUND(AVG(k.completeness_rate), 2)                                      AS avg_completeness_rate,
    ROUND(AVG(k.effectiveness_score), 2)                                    AS avg_effectiveness_score,
    ROUND(AVG(k.engagement_score), 2)                                       AS avg_engagement_score,
    ROUND(AVG(k.skill_gap_index), 2)                                        AS avg_skill_gap_index,
    SUM(k.verified_certs)                                                   AS total_verified_certs,
    SUM(k.total_courses)                                                    AS total_courses_enrolled,
    SUM(k.completed_courses)                                                AS total_courses_completed,
    SUM(CASE WHEN k.voucher_eligibility LIKE '100%' THEN 1 ELSE 0 END)     AS employees_100pct_voucher,
    SUM(CASE WHEN k.voucher_eligibility = '50% Discount' THEN 1 ELSE 0 END) AS employees_50pct_voucher,
    SUM(CASE WHEN k.voucher_eligibility = 'Not Eligible' THEN 1 ELSE 0 END) AS employees_not_eligible,
    CURRENT_TIMESTAMP()                                                     AS _view_refreshed_ts
FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360 k
JOIN {CATALOG}.{GOLD_SCHEMA}.dim_employee e ON k.employee_id = e.employee_id
WHERE e.is_current = true
GROUP BY e.department
""")

print("✅ vw_department_summary created")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 12 — VIEW: vw_course_effectiveness

# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_course_effectiveness AS

SELECT
    f.course_id,
    c.course_name,
    c.category,
    c.vendor,
    c.delivery_mode,
    c.duration_hours,
    c.difficulty_level,
    COUNT(DISTINCT f.employee_id)                                                        AS total_learners,
    SUM(CASE WHEN f.is_completed = true THEN 1 ELSE 0 END)                               AS total_completed,
    ROUND(SUM(CASE WHEN f.is_completed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS completion_rate_pct,
    SUM(CASE WHEN f.is_passed = true THEN 1 ELSE 0 END)                                  AS total_passed,
    ROUND(SUM(CASE WHEN f.is_passed = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)    AS pass_rate_pct,
    ROUND(AVG(CASE WHEN f.is_passed = true THEN f.score END), 2)                         AS avg_score_passed,
    ROUND(AVG(f.feedback_rating), 2)                                                     AS avg_feedback_rating,
    ROUND(AVG(f.days_to_complete), 2)                                                    AS avg_days_to_complete,
    ROUND(AVG(f.total_learning_days), 2)                                                 AS avg_total_learning_days,
    COUNT(DISTINCT f.trainer_name)                                                       AS total_trainers,
    CURRENT_TIMESTAMP()                                                                  AS _view_refreshed_ts
FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity f
JOIN {CATALOG}.{GOLD_SCHEMA}.dim_course c ON f.course_id = c.course_id
GROUP BY f.course_id, c.course_name, c.category, c.vendor,
         c.delivery_mode, c.duration_hours, c.difficulty_level
""")

print("✅ vw_course_effectiveness created")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 13 — VIEW: vw_voucher_eligibility

# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_voucher_eligibility AS

SELECT
    employee_id,
    employee_name,
    department,
    designation,
    grade,
    manager_name,
    completeness_rate,
    effectiveness_score,
    engagement_score,
    eng_sig_completion_rate,
    eng_sig_pass_rate,
    eng_sig_avg_score,
    eng_sig_feedback,
    eng_sig_speed,
    verified_certs,
    is_first_time_voucher,
    voucher_eligibility,
    CASE
        WHEN voucher_eligibility LIKE '100%' THEN 1
        WHEN voucher_eligibility = '50% Discount'  THEN 2
        ELSE 3
    END                             AS voucher_priority_rank,
    _view_refreshed_ts
FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360
ORDER BY voucher_priority_rank, completeness_rate DESC
""")

print("✅ vw_voucher_eligibility created")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 14 — VIEW: vw_agent_context_full (Master Agent View)

# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_agent_context_full AS

WITH skill_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT primary_skill))   AS skills_list,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT skill_category))  AS skill_categories,
        COUNT(DISTINCT primary_skill)                           AS distinct_skill_count,
        SUM(CASE WHEN skill_readiness_status = 'Fully Ready' THEN 1 ELSE 0 END) AS ready_skill_count
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_skill
    GROUP BY employee_id
),

cert_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT certification_name)) AS certifications_list,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT vendor))              AS cert_vendors,
        COUNT(certification_id)                                     AS total_certs,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)         AS verified_cert_count
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_certification
    GROUP BY employee_id
),

course_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT course_id))      AS completed_course_ids,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT course_domain))  AS course_domains
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    WHERE is_completed = true
    GROUP BY employee_id
)

SELECT
    k.employee_id,
    k.employee_name,
    k.department,
    k.designation,
    k.grade,
    k.location,
    k.manager_name,
    k.is_manager,
    k.is_fresher,
    k.total_prior_exp_years,
    k.current_tenure_years,
    k.employee_scd_version,

    -- KPIs
    k.completeness_rate,
    k.effectiveness_score,
    k.engagement_score,
    k.skill_gap_index,
    k.verified_certs,

    -- Engagement breakdown
    k.eng_sig_completion_rate,
    k.eng_sig_pass_rate,
    k.eng_sig_avg_score,
    k.eng_sig_feedback,
    k.eng_sig_speed,

    -- Learning counts
    k.total_courses,
    k.completed_courses,
    k.passed_courses,
    k.avg_learning_days,
    k.avg_days_to_complete,

    -- Skills
    COALESCE(sk.skills_list, 'None')        AS skills,
    COALESCE(sk.skill_categories, 'None')   AS skill_categories,
    COALESCE(sk.distinct_skill_count, 0)    AS distinct_skill_count,
    COALESCE(sk.ready_skill_count, 0)       AS ready_skill_count,

    -- Certifications
    COALESCE(ct.certifications_list, 'None') AS certifications,
    COALESCE(ct.cert_vendors, 'None')        AS cert_vendors,
    COALESCE(ct.total_certs, 0)              AS total_certs,
    COALESCE(ct.verified_cert_count, 0)      AS verified_cert_count,

    -- Completed courses
    COALESCE(ca.completed_course_ids, 'None') AS completed_course_ids,
    COALESCE(ca.course_domains, 'None')       AS completed_course_domains,

    -- Voucher
    k.is_first_time_voucher,
    k.voucher_eligibility,

    -- LLM-ready context text (injected directly into agent prompt)
    CONCAT(
        k.employee_name, ' (', k.employee_id, ') is a ', k.designation,
        ' in ', k.department, ' at grade ', k.grade,
        ', reporting to ', k.manager_name, '. ',
        'Tenure: ', ROUND(k.current_tenure_years, 1), ' yrs. ',
        'Completed ', k.completed_courses, '/', k.total_courses, ' courses. ',
        'Effectiveness score: ', k.effectiveness_score, '/100. ',
        'Engagement score: ', k.engagement_score, '/100 ',
        '(completion=', k.eng_sig_completion_rate,
        '%, pass=', k.eng_sig_pass_rate,
        '%, score=', k.eng_sig_avg_score,
        ', feedback=', k.eng_sig_feedback,
        ', speed=', k.eng_sig_speed, '). ',
        'Skill gap index: ', k.skill_gap_index, '%. ',
        'Verified certs: ', k.verified_certs, '. ',
        'Skills: ', COALESCE(sk.skills_list, 'None'), '. ',
        'Certifications: ', COALESCE(ct.certifications_list, 'None'), '. ',
        'First-time voucher applicant: ', k.is_first_time_voucher, '. ',
        'Voucher eligibility: ', k.voucher_eligibility, '.'
    )                                        AS agent_context_text,

    k._view_refreshed_ts

FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360 k
LEFT JOIN skill_agg  sk ON k.employee_id = sk.employee_id
LEFT JOIN cert_agg   ct ON k.employee_id = ct.employee_id
LEFT JOIN course_agg ca ON k.employee_id = ca.employee_id
""")

print("✅ vw_agent_context_full (v2) created")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 15 — Verification

# COMMAND ----------

print("=" * 60)
print("GOLD LAYER v2 — TABLE & VIEW INVENTORY")
print("=" * 60)

tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").collect()
for t in tables:
    count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{t['tableName']}").count()
    icon = "📊" if "fact" in t["tableName"] else "📋" if "dim" in t["tableName"] else "👁️ "
    print(f"  {icon} {t['tableName']:<45} {count:>6} rows")

print("=" * 60)
print("\n📋 dim_employee — SCD Type 2 version check:")
spark.sql(f"""
    SELECT employee_id, employee_name, designation, department,
           valid_from, valid_to, is_current, scd_version
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_employee
    ORDER BY employee_id, scd_version
    LIMIT 5
""").show(truncate=40)

print("\n🎟️  Voucher eligibility breakdown:")
spark.sql(f"""
    SELECT voucher_eligibility, is_first_time_voucher,
           COUNT(*) AS employee_count,
           ROUND(AVG(engagement_score), 1) AS avg_engagement,
           ROUND(AVG(completeness_rate), 1) AS avg_completeness
    FROM {CATALOG}.{GOLD_SCHEMA}.vw_voucher_eligibility
    GROUP BY voucher_eligibility, is_first_time_voucher
    ORDER BY CASE
        WHEN voucher_eligibility LIKE '100%' THEN 1
        WHEN voucher_eligibility = '50% Discount' THEN 2
        ELSE 3
    END
""").show(truncate=50)

print("\n🤖 Sample Agent Context (2 employees):")
spark.sql(f"""
    SELECT employee_id, employee_name, engagement_score,
           is_first_time_voucher, voucher_eligibility, agent_context_text
    FROM {CATALOG}.{GOLD_SCHEMA}.vw_agent_context_full
    LIMIT 2
""").show(truncate=100)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 16 — Helper: Record a voucher issuance
# MAGIC
# MAGIC Call this function whenever the agent actually issues a voucher.
# MAGIC It appends a row to `dim_voucher_history` so subsequent runs correctly
# MAGIC identify this employee as a non-first-timer.

# COMMAND ----------

import uuid
from datetime import date

def insert_voucher_event(employee_id: str, voucher_type: str,
                         trigger_rule: str, course_id: str = None,
                         issued_by: str = "agent"):
    """
    Records a voucher issuance in dim_voucher_history.
    Call this after the agent confirms a voucher is granted.
    """
    row = [(
        str(uuid.uuid4()),
        employee_id,
        date.today(),
        voucher_type,
        trigger_rule,
        course_id,
        issued_by,
        __import__("datetime").datetime.now()
    )]
    schema = StructType([
        StructField("voucher_id",       StringType(),    False),
        StructField("employee_id",      StringType(),    False),
        StructField("voucher_date",     DateType(),      False),
        StructField("voucher_type",     StringType(),    False),
        StructField("trigger_rule",     StringType(),    True),
        StructField("course_id",        StringType(),    True),
        StructField("issued_by",        StringType(),    True),
        StructField("_gold_created_ts", TimestampType(), True),
    ])
    spark.createDataFrame(row, schema) \
         .write.format("delta").mode("append") \
         .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.dim_voucher_history")
    print(f"✅ Voucher recorded: {employee_id} | {voucher_type} | rule={trigger_rule}")

# Example (uncomment to test):
# insert_voucher_event("E10001", "100% Free", "First Timer", course_id="C1001_DB")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 17 — Optional: Persist agent training dataset

# COMMAND ----------

# Uncomment to materialise the master agent view as a physical Delta table for fine-tuning

# (
#     spark.table(f"{CATALOG}.{GOLD_SCHEMA}.vw_agent_context_full")
#     .write.format("delta")
#     .mode("overwrite").option("overwriteSchema", "true")
#     .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.agent_training_dataset")
# )
# print("✅ agent_training_dataset exported")

✅ Using catalog: hackathon_ltm.gold
✅ All 7 Silver tables loaded.
✅ dim_employee: no new employees to insert
✅ dim_course: 490 rows
✅ dim_date: 1922 rows
✅ dim_skill: 454 rows
✅ dim_certification: 400 rows
✅ dim_voucher_history: 0 prior vouchers on record
✅ fact_learning_activity (MERGE): 660 total rows
✅ vw_employee_kpi_360 (v2) created
✅ vw_department_summary created
✅ vw_course_effectiveness created
✅ vw_voucher_eligibility created
✅ vw_agent_context_full (v2) created
GOLD LAYER v2 — TABLE & VIEW INVENTORY
  📋 dim_certification                                400 rows
  📋 dim_course                                       490 rows
  📋 dim_date                                        1922 rows
  📋 dim_employee                                     476 rows
  📋 dim_skill                                        454 rows
  📋 dim_voucher_history                                0 rows
  📊 fact_learning_activity                           660 rows
  👁️  vw_agent_context_full                        

In [0]:
%sql
select * from hackathon_ltm.gold.vw_voucher_eligibility

employee_id,employee_name,department,designation,grade,manager_name,completeness_rate,effectiveness_score,engagement_score,eng_sig_completion_rate,eng_sig_pass_rate,eng_sig_avg_score,eng_sig_feedback,eng_sig_speed,verified_certs,is_first_time_voucher,voucher_eligibility,voucher_priority_rank,_view_refreshed_ts
E10188,Meera Singh,Data Engineering,Trainee,P1,Amit Desai,100.00,90.0,73.6,100.00,100.00,90.0,0.0,6.0,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10184,Kavya Jain,Domain Consulting,Consultant,P1,Meera Reddy,100.00,86.0,76.35,100.00,100.00,86.0,0.0,41.5,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10230,Amit Chatterjee,Digital Engineering,Software Engineer,P1,Anjali Patel,100.00,94.0,75.7,100.00,100.00,94.0,0.0,19.0,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10146,Kavya Verma,Cloud & Platform,Trainee,P1,Vikram Rao,100.00,94.0,87.9,100.00,100.00,94.0,90.0,6.0,0,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10177,Reyansh Bose,Ai & Data Science,Data Scientist,P1,Neha Singh,100.00,81.0,71.8,100.00,100.00,81.0,0.0,6.0,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10240,Aditya Shah,Bi & Analytics,Contractor,P1,Priya Sharma,100.00,77.0,72.7,100.00,100.00,77.0,0.0,23.0,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10142,Pooja Iyer,Bi & Analytics,Senior Analyst,P2,Karan Gupta,100.00,88.0,89.05,100.00,100.00,88.0,84.0,38.5,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10211,Isha Agarwal,Cloud & Platform,Cloud Engineer,P1,Vikram Rao,100.00,85.0,77.75,100.00,100.00,85.0,0.0,57.5,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10126,Kavya Pandey,Bi & Analytics,Analyst,P1,Karan Gupta,100.00,78.0,84.45,100.00,100.00,78.0,88.0,6.5,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z
E10246,Atharva Gupta,Digital Engineering,Senior Engineer,P2,Anjali Patel,100.00,78.0,71.1,100.00,100.00,78.0,0.0,5.0,1,true,100% Free — First Timer,1,2026-06-10T12:56:37.672Z


In [0]:

CATALOG     = "hackathon_ltm"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {GOLD_SCHEMA}")

print(f"✅ Using {CATALOG}.{GOLD_SCHEMA}")



spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360 AS

WITH learning_stats AS (
    SELECT
        employee_id,
        COUNT(learning_id)                                                              AS total_courses,
        SUM(CASE WHEN completion_status = 'Completed' THEN 1 ELSE 0 END)               AS completed_courses,
        SUM(CASE WHEN is_passed = true THEN 1 ELSE 0 END)                              AS passed_courses,
        ROUND(AVG(CASE WHEN is_passed = true THEN score END), 2)                       AS avg_score_passed,
        ROUND(AVG(total_learning_days), 2)                                             AS avg_learning_days,
        ROUND(AVG(days_to_complete), 2)                                                AS avg_days_to_complete,
        ROUND(AVG(CASE WHEN is_passed = true THEN score ELSE 0 END), 2)                AS avg_score_all
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    GROUP BY employee_id
),

feedback_stats AS (
    SELECT
        employee_id,
        ROUND(AVG(feedback_rating), 2)                                                 AS avg_feedback_rating,
        SUM(CASE WHEN feedback_recommended = true THEN 1 ELSE 0 END)                   AS total_recommended
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    WHERE feedback_rating IS NOT NULL
    GROUP BY employee_id
),

course_duration_stats AS (
    SELECT
        f.employee_id,
        ROUND(
            AVG(
                LEAST(
                    COALESCE(c.duration_hours, 1) / NULLIF(f.days_to_complete, 0),
                    2.0
                )
            ),
        2) AS avg_completion_speed_index
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity f
    JOIN {CATALOG}.{GOLD_SCHEMA}.dim_course c ON f.course_id = c.course_id
    WHERE f.is_completed = true AND f.days_to_complete > 0
    GROUP BY f.employee_id
),

skill_stats AS (
    SELECT
        employee_id,
        COUNT(skill_id)                                                                AS total_skills,
        SUM(CASE WHEN skill_readiness_status = 'Fully Ready' THEN 1 ELSE 0 END)        AS fully_ready_skills,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)                            AS verified_skills
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_skill
    GROUP BY employee_id
),

cert_stats AS (
    SELECT
        employee_id,
        COUNT(certification_id)                                                        AS total_certs,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)                            AS verified_certs
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_certification
    GROUP BY employee_id
),

-- Weighted engagement score (5 signals) — unchanged from v2
engagement_calc AS (
    SELECT
        l.employee_id,

        ROUND(COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2)
            AS sig_completion_rate,

        ROUND(COALESCE(l.passed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2)
            AS sig_pass_rate,

        ROUND(COALESCE(l.avg_score_all, 0), 2)
            AS sig_avg_score,

        ROUND(COALESCE(f.avg_feedback_rating, 0) * 20.0, 2)
            AS sig_feedback_normalised,

        ROUND(LEAST(COALESCE(cs.avg_completion_speed_index, 1.0), 2.0) / 2.0 * 100.0, 2)
            AS sig_speed_normalised,

        ROUND(
              COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0)  * 0.30
            + COALESCE(l.passed_courses    * 100.0 / NULLIF(l.total_courses, 0), 0)  * 0.25
            + COALESCE(l.avg_score_all, 0)                                            * 0.20
            + COALESCE(f.avg_feedback_rating, 0) * 20.0                              * 0.15
            + LEAST(COALESCE(cs.avg_completion_speed_index, 1.0), 2.0) / 2.0 * 100.0 * 0.10
        , 2) AS engagement_score
    FROM learning_stats l
    LEFT JOIN feedback_stats f  ON l.employee_id = f.employee_id
    LEFT JOIN course_duration_stats cs ON l.employee_id = cs.employee_id
)

SELECT
    e.employee_id,
    e.employee_name,
    e.department,
    e.designation,
    e.grade,
    e.location,
    e.manager_name,
    e.current_tenure_years,
    e.is_manager,
    e.is_fresher,
    e.total_prior_exp_years,
    e.valid_from                                                                        AS employee_version_from,
    e.scd_version                                                                       AS employee_scd_version,

    -- KPI 1: Completeness Rate (0-100)
    ROUND(COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2)    AS completeness_rate,

    -- KPI 2: Effectiveness Score (avg score on passed courses, 0-100)
    COALESCE(l.avg_score_passed, 0)                                                    AS effectiveness_score,

    -- KPI 3: Engagement Score (weighted composite, 0-100)
    COALESCE(eg.engagement_score, 0)                                                   AS engagement_score,
    COALESCE(eg.sig_completion_rate, 0)                                                AS eng_sig_completion_rate,
    COALESCE(eg.sig_pass_rate, 0)                                                      AS eng_sig_pass_rate,
    COALESCE(eg.sig_avg_score, 0)                                                      AS eng_sig_avg_score,
    COALESCE(eg.sig_feedback_normalised, 0)                                            AS eng_sig_feedback,
    COALESCE(eg.sig_speed_normalised, 0)                                               AS eng_sig_speed,

    -- KPI 4: Skill Gap Index — v3: RESCALED to 0-1 (100 -> 1, 0 -> 0)
    ROUND(
        (100.0 - COALESCE(s.fully_ready_skills * 100.0 / NULLIF(s.total_skills, 0), 0)) / 100.0
    , 2)                                                                                AS skill_gap_index,

    -- KPI 5: Verified Certifications
    COALESCE(c.verified_certs, 0)                                                      AS verified_certs,

    -- Supporting counts
    COALESCE(l.total_courses, 0)                                                       AS total_courses,
    COALESCE(l.completed_courses, 0)                                                   AS completed_courses,
    COALESCE(l.passed_courses, 0)                                                      AS passed_courses,
    COALESCE(s.total_skills, 0)                                                        AS total_skills,
    COALESCE(s.fully_ready_skills, 0)                                                  AS fully_ready_skills,
    COALESCE(s.verified_skills, 0)                                                     AS verified_skills,
    COALESCE(c.total_certs, 0)                                                         AS total_certs,
    COALESCE(l.avg_learning_days, 0)                                                   AS avg_learning_days,
    COALESCE(l.avg_days_to_complete, 0)                                                AS avg_days_to_complete,
    COALESCE(f.total_recommended, 0)                                                   AS courses_recommended,

    -- ── v3: is_first_time_voucher — KPI-based, no longer uses dim_voucher_history ──
    -- TRUE if passed_courses is 0 or 1 AND total_certs = 0
    CASE
        WHEN COALESCE(l.passed_courses, 0) IN (0, 1)
         AND COALESCE(c.total_certs, 0) = 0
        THEN true
        ELSE false
    END                                                                                 AS is_first_time_voucher,

    -- ── v3: voucher_eligibility — binary Eligible / Not Eligible ──
    CASE
        -- Rule 0: First timer -> automatically Eligible
        WHEN (
            CASE
                WHEN COALESCE(l.passed_courses, 0) IN (0, 1)
                 AND COALESCE(c.total_certs, 0) = 0
                THEN true
                ELSE false
            END
        ) = true
            THEN 'Eligible'

        -- Rule 1: Not a first timer -> check completeness, effectiveness, engagement
        WHEN ROUND(COALESCE(l.completed_courses * 100.0 / NULLIF(l.total_courses, 0), 0), 2) = 100
         AND COALESCE(l.avg_score_passed, 0) >= 75
         AND COALESCE(eg.engagement_score, 0) > 50
            THEN 'Eligible'

        ELSE 'Not Eligible'
    END                                                                                 AS voucher_eligibility,

    CURRENT_TIMESTAMP()                                                                AS _view_refreshed_ts

FROM {CATALOG}.{GOLD_SCHEMA}.dim_employee e
LEFT JOIN learning_stats        l  ON e.employee_id = l.employee_id
LEFT JOIN feedback_stats        f  ON e.employee_id = f.employee_id
LEFT JOIN engagement_calc       eg ON e.employee_id = eg.employee_id
LEFT JOIN skill_stats           s  ON e.employee_id = s.employee_id
LEFT JOIN cert_stats            c  ON e.employee_id = c.employee_id
WHERE e.is_current = true
""")

print("✅ vw_employee_kpi_360 (v3) created")


spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_voucher_eligibility AS

SELECT
    employee_id,
    employee_name,
    department,
    designation,
    grade,
    manager_name,
    completeness_rate,
    effectiveness_score,
    engagement_score,
    eng_sig_completion_rate,
    eng_sig_pass_rate,
    eng_sig_avg_score,
    eng_sig_feedback,
    eng_sig_speed,
    skill_gap_index,
    verified_certs,
    total_certs,
    passed_courses,
    is_first_time_voucher,
    voucher_eligibility,
    CASE
        WHEN voucher_eligibility = 'Eligible' THEN 1
        ELSE 2
    END                             AS voucher_priority_rank,
    _view_refreshed_ts
FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360
ORDER BY voucher_priority_rank, completeness_rate DESC
""")

print("✅ vw_voucher_eligibility (v3) created")

-

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{GOLD_SCHEMA}.vw_agent_context_full AS

WITH skill_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT primary_skill))   AS skills_list,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT skill_category))  AS skill_categories,
        COUNT(DISTINCT primary_skill)                           AS distinct_skill_count,
        SUM(CASE WHEN skill_readiness_status = 'Fully Ready' THEN 1 ELSE 0 END) AS ready_skill_count
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_skill
    GROUP BY employee_id
),

cert_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT certification_name)) AS certifications_list,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT vendor))              AS cert_vendors,
        COUNT(certification_id)                                     AS total_certs,
        SUM(CASE WHEN is_verified = true THEN 1 ELSE 0 END)         AS verified_cert_count
    FROM {CATALOG}.{GOLD_SCHEMA}.dim_certification
    GROUP BY employee_id
),

course_agg AS (
    SELECT
        employee_id,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT course_id))      AS completed_course_ids,
        CONCAT_WS(', ', COLLECT_LIST(DISTINCT course_domain))  AS course_domains
    FROM {CATALOG}.{GOLD_SCHEMA}.fact_learning_activity
    WHERE is_completed = true
    GROUP BY employee_id
)

SELECT
    k.employee_id,
    k.employee_name,
    k.department,
    k.designation,
    k.grade,
    k.location,
    k.manager_name,
    k.is_manager,
    k.is_fresher,
    k.total_prior_exp_years,
    k.current_tenure_years,
    k.employee_scd_version,

    -- KPIs
    k.completeness_rate,
    k.effectiveness_score,
    k.engagement_score,
    k.skill_gap_index,           -- v3: now 0-1 scale
    k.verified_certs,

    -- Engagement breakdown
    k.eng_sig_completion_rate,
    k.eng_sig_pass_rate,
    k.eng_sig_avg_score,
    k.eng_sig_feedback,
    k.eng_sig_speed,

    -- Learning counts
    k.total_courses,
    k.completed_courses,
    k.passed_courses,
    k.avg_learning_days,
    k.avg_days_to_complete,

    -- Skills
    COALESCE(sk.skills_list, 'None')        AS skills,
    COALESCE(sk.skill_categories, 'None')   AS skill_categories,
    COALESCE(sk.distinct_skill_count, 0)    AS distinct_skill_count,
    COALESCE(sk.ready_skill_count, 0)       AS ready_skill_count,

    -- Certifications
    COALESCE(ct.certifications_list, 'None') AS certifications,
    COALESCE(ct.cert_vendors, 'None')        AS cert_vendors,
    COALESCE(ct.total_certs, 0)              AS total_certs,
    COALESCE(ct.verified_cert_count, 0)      AS verified_cert_count,

    -- Completed courses
    COALESCE(ca.completed_course_ids, 'None') AS completed_course_ids,
    COALESCE(ca.course_domains, 'None')       AS completed_course_domains,

    -- Voucher (v3: binary)
    k.is_first_time_voucher,
    k.voucher_eligibility,

    -- LLM-ready context text (v3 wording updated for skill_gap_index scale + binary voucher)
    CONCAT(
        k.employee_name, ' (', k.employee_id, ') is a ', k.designation,
        ' in ', k.department, ' at grade ', k.grade,
        ', reporting to ', k.manager_name, '. ',
        'Tenure: ', ROUND(k.current_tenure_years, 1), ' yrs. ',
        'Completed ', k.completed_courses, '/', k.total_courses, ' courses. ',
        'Effectiveness score: ', k.effectiveness_score, '/100. ',
        'Engagement score: ', k.engagement_score, '/100 ',
        '(completion=', k.eng_sig_completion_rate,
        '%, pass=', k.eng_sig_pass_rate,
        '%, score=', k.eng_sig_avg_score,
        ', feedback=', k.eng_sig_feedback,
        ', speed=', k.eng_sig_speed, '). ',
        'Skill gap index: ', k.skill_gap_index, ' (0=no gap, 1=full gap). ',
        'Verified certs: ', k.verified_certs, '. ',
        'Skills: ', COALESCE(sk.skills_list, 'None'), '. ',
        'Certifications: ', COALESCE(ct.certifications_list, 'None'), '. ',
        'First-time voucher applicant: ', k.is_first_time_voucher, '. ',
        'Voucher eligibility: ', k.voucher_eligibility, '.'
    )                                        AS agent_context_text,

    k._view_refreshed_ts

FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360 k
LEFT JOIN skill_agg  sk ON k.employee_id = sk.employee_id
LEFT JOIN cert_agg   ct ON k.employee_id = ct.employee_id
LEFT JOIN course_agg ca ON k.employee_id = ca.employee_id
""")

print("✅ vw_agent_context_full (v3) created")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell E — Verification

# COMMAND ----------

print("=" * 60)
print("GOLD LAYER v3 — VERIFICATION")
print("=" * 60)

print("\n📐 skill_gap_index rescale check (should be 0.00-1.00):")
spark.sql(f"""
    SELECT employee_id, total_skills, fully_ready_skills, skill_gap_index
    FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360
    ORDER BY skill_gap_index DESC
    LIMIT 5
""").show(truncate=False)

print("\n🎟️  is_first_time_voucher logic check:")
spark.sql(f"""
    SELECT
        is_first_time_voucher,
        COUNT(*) AS employee_count,
        ROUND(AVG(passed_courses), 2) AS avg_passed_courses,
        ROUND(AVG(total_certs), 2)    AS avg_total_certs
    FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360
    GROUP BY is_first_time_voucher
""").show(truncate=False)

print("\n🎟️  voucher_eligibility breakdown:")
spark.sql(f"""
    SELECT
        is_first_time_voucher,
        voucher_eligibility,
        COUNT(*) AS employee_count,
        ROUND(AVG(completeness_rate), 1)  AS avg_completeness,
        ROUND(AVG(effectiveness_score), 1) AS avg_effectiveness,
        ROUND(AVG(engagement_score), 1)    AS avg_engagement
    FROM {CATALOG}.{GOLD_SCHEMA}.vw_employee_kpi_360
    GROUP BY is_first_time_voucher, voucher_eligibility
    ORDER BY is_first_time_voucher, voucher_eligibility
""").show(truncate=False)

print("\n🤖 Sample agent context (3 employees):")
spark.sql(f"""
    SELECT employee_id, employee_name, skill_gap_index,
           is_first_time_voucher, voucher_eligibility, agent_context_text
    FROM {CATALOG}.{GOLD_SCHEMA}.vw_agent_context_full
    LIMIT 3
""").show(truncate=120)

✅ Using hackathon_ltm.gold
✅ vw_employee_kpi_360 (v3) created
✅ vw_voucher_eligibility (v3) created
✅ vw_agent_context_full (v3) created
GOLD LAYER v3 — VERIFICATION

📐 skill_gap_index rescale check (should be 0.00-1.00):
+-----------+------------+------------------+---------------+
|employee_id|total_skills|fully_ready_skills|skill_gap_index|
+-----------+------------+------------------+---------------+
|E10377     |0           |0                 |1.00           |
|E10454     |1           |0                 |1.00           |
|E10033     |1           |0                 |1.00           |
|E10339     |0           |0                 |1.00           |
|E10185     |0           |0                 |1.00           |
+-----------+------------+------------------+---------------+


🎟️  is_first_time_voucher logic check:
+---------------------+--------------+------------------+---------------+
|is_first_time_voucher|employee_count|avg_passed_courses|avg_total_certs|
+---------------------+--------

In [0]:
%sql
select * from hackathon_ltm.gold.vw_agent_context_full

employee_id,employee_name,department,designation,grade,location,manager_name,is_manager,is_fresher,total_prior_exp_years,current_tenure_years,employee_scd_version,completeness_rate,effectiveness_score,engagement_score,skill_gap_index,verified_certs,eng_sig_completion_rate,eng_sig_pass_rate,eng_sig_avg_score,eng_sig_feedback,eng_sig_speed,total_courses,completed_courses,passed_courses,avg_learning_days,avg_days_to_complete,skills,skill_categories,distinct_skill_count,ready_skill_count,certifications,cert_vendors,total_certs,verified_cert_count,completed_course_ids,completed_course_domains,is_first_time_voucher,voucher_eligibility,agent_context_text,_view_refreshed_ts
E10339,Aarav Chatterjee,Ai & Data Science,Senior Data Scientist,P2,Mumbai,Neha Singh,false,false,5.0,6.65,1,0.00,0.0,0.0,1.00,0,0.00,0.00,0.0,0.0,0.0,0,0,0,0.0,0.0,None,None,0,0,None,None,0,0,None,None,true,Eligible,"Aarav Chatterjee (E10339) is a Senior Data Scientist in Ai & Data Science at grade P2, reporting to Neha Singh. Tenure: 6.7 yrs. Completed 0/0 courses. Effectiveness score: 0.0/100. Engagement score: 0.0/100 (completion=0.00%, pass=0.00%, score=0.0, feedback=0.0, speed=0.0). Skill gap index: 1.00 (0=no gap, 1=full gap). Verified certs: 0. Skills: None. Certifications: None. First-time voucher applicant: true. Voucher eligibility: Eligible.",2026-06-13T14:39:46.238Z
E10224,Aarav Iyer,Bi & Analytics,Analyst,P1,Mumbai,Priya Sharma,false,false,2.02,4.49,1,100.00,90.0,77.35,0.00,1,100.00,100.00,90.0,0.0,43.5,1,1,1,544.0,23.0,Machine Learning,Data & Analytics,1,1,Enterprise Agile Frameworks Certificate,Coursera,1,1,C1431_CR,Core / Compliance,false,Eligible,"Aarav Iyer (E10224) is a Analyst in Bi & Analytics at grade P1, reporting to Priya Sharma. Tenure: 4.5 yrs. Completed 1/1 courses. Effectiveness score: 90.0/100. Engagement score: 77.35/100 (completion=100.00%, pass=100.00%, score=90.0, feedback=0.0, speed=43.5). Skill gap index: 0.00 (0=no gap, 1=full gap). Verified certs: 1. Skills: Machine Learning. Certifications: Enterprise Agile Frameworks Certificate. First-time voucher applicant: false. Voucher eligibility: Eligible.",2026-06-13T14:39:46.238Z
E10454,Aarav Jain,Cloud & Platform,Senior Cloud Engineer,P2,Mumbai,Vikram Rao,false,false,5.04,7.12,1,0.00,0.0,0.0,1.00,0,0.00,0.00,0.0,0.0,0.0,0,0,0,0.0,0.0,SAP,Enterprise Apps,1,0,None,None,0,0,None,None,true,Eligible,"Aarav Jain (E10454) is a Senior Cloud Engineer in Cloud & Platform at grade P2, reporting to Vikram Rao. Tenure: 7.1 yrs. Completed 0/0 courses. Effectiveness score: 0.0/100. Engagement score: 0.0/100 (completion=0.00%, pass=0.00%, score=0.0, feedback=0.0, speed=0.0). Skill gap index: 1.00 (0=no gap, 1=full gap). Verified certs: 0. Skills: SAP. Certifications: None. First-time voucher applicant: true. Voucher eligibility: Eligible.",2026-06-13T14:39:46.238Z
E10033,Aarav Joshi,Data Engineering,Contractor,P1,Pune,Sneha Iyer,false,false,2.25,3.55,1,50.00,85.0,49.1,1.00,2,50.00,50.00,42.5,84.0,5.0,4,2,2,84.0,25.0,Databricks,Data & Analytics,1,0,"Databricks Lakehouse Fundamentals Certification, Azure Data Fundamentals DP-900","Databricks, Microsoft",2,2,C1001_DB,Data & BI,false,Not Eligible,"Aarav Joshi (E10033) is a Contractor in Data Engineering at grade P1, reporting to Sneha Iyer. Tenure: 3.6 yrs. Completed 2/4 courses. Effectiveness score: 85.0/100. Engagement score: 49.1/100 (completion=50.00%, pass=50.00%, score=42.5, feedback=84.0, speed=5.0). Skill gap index: 1.00 (0=no gap, 1=full gap). Verified certs: 2. Skills: Databricks. Certifications: Databricks Lakehouse Fundamentals Certification, Azure Data Fundamentals DP-900. First-time voucher applicant: false. Voucher eligibility: Not Eligible.",2026-06-13T14:39:46.238Z
E10377,Aarav Mehta,Enterprise Applications,Developer,P1,Noida,Rohan Joshi,false,false,2.04,4.79,1,0.00,0.0,0.0,1.00,0,0.00,0.00,0.0,0.0,0.0,0,0,0,0.0,0.0,None,None,0,0,None,None,0,0,None,None,true,Eligible,"Aarav Mehta (E10377) is a Developer in Enterprise Applications at gra